# Agent Orchestrateur – LangGraph SMA

## Rôle et Architecture

Ce notebook implémente l'**orchestrateur central** du Système Multi-Agents (SMA) en utilisant **LangGraph**. Il coordonne l'exécution des trois agents spécialisés et implémente un workflow stateful avec :

### Composants du SMA

| Agent | Rôle | Modèle |
|-------|------|--------|
| Agent 1 – Ingestion | Extraction PDF → ChromaDB | sentence-transformers (local) |
| Agent 2 – Synthèse | RAG + Analyse thématique | claude-sonnet-4-5 |
| Agent 3 – Gaps | Détection Chain-of-Thought | claude-sonnet-4-5 |
| **Orchestrateur** | Coordination + HITL | LangGraph StateGraph |

### Fonctionnalités de l'Orchestrateur

1. **State Management** : État partagé entre tous les agents via `AgentState`
2. **Human-in-the-Loop (HITL)** : Validation humaine avant le rapport final
3. **Conditional Routing** : Routage conditionnel selon l'approbation ou les erreurs
4. **Fallback Mechanism** : Récupération automatique en cas d'erreur
5. **Recursion Limit** : Protection contre les boucles infinies

### Graphe de flux
```
START → [Ingestion] → [Synthèse] → [Gap Detection] → [Human Validation]
                                         ↑                    |
                                         |       Refus         |
                                         └────────────────────┘
                                                   | Approbation
                                                   ↓
                                          [Export Report] → END
```

In [1]:
# ============================================================
# Installation des dépendances
# ============================================================
# !pip install langgraph anthropic python-dotenv
# !pip install pdfplumber chromadb sentence-transformers langchain-text-splitters

# ============================================================
# Imports
# ============================================================
import sys
import os
import json
import time
import operator
from pathlib import Path
from datetime import datetime
from typing import TypedDict, Annotated, Literal

# LangGraph
from langgraph.graph import StateGraph, END, START

# Anthropic
import llm_adapter as anthropic  # adaptateur Gemini (gratuit)

# Variables d'environnement
from dotenv import load_dotenv

# Charger les variables d'environnement
load_dotenv(dotenv_path=Path("../.env"))
ANTHROPIC_API_KEY = os.getenv("GEMINI_API_KEY", "")

print("Imports réussis.")
print(f"Anthropic version : {anthropic.__version__}")

# Vérification de la clé API
if ANTHROPIC_API_KEY:
    print(f"Clé API Anthropic : {ANTHROPIC_API_KEY[:8]}...")
else:
    print("[ATTENTION] ANTHROPIC_API_KEY manquante dans .env")

# Constantes globales
LOGS_DIR   = "../logs"
RAPPORT_DIR = "../Rapport"

Imports réussis.
Anthropic version : llm_adapter/gemini-2.0-flash
Clé API Anthropic : AIza****...


In [2]:
# ============================================================
# Définition de l'état partagé du SMA (AgentState)
# ============================================================

class AgentState(TypedDict):
    """État partagé entre tous les agents du SMA.
    
    Cet état est le 'registre central' du système : chaque agent
    lit les résultats des agents précédents et y écrit ses propres
    résultats. LangGraph garantit la persistance de cet état
    entre chaque nœud du graphe.
    """
    domain: str
    """Domaine de recherche fourni par l'utilisateur au démarrage."""

    ingestion_result: dict
    """Résultat de l'Agent 1 : statistiques d'ingestion et liste des articles."""

    synthesis_result: dict
    """Résultat de l'Agent 2 : synthèse thématique structurée par thème."""

    gaps_result: dict
    """Résultat de l'Agent 3 : gaps de recherche détectés et rapport."""

    human_approved: bool
    """Indicateur Human-in-the-Loop : True si l'utilisateur a validé les gaps."""

    error_count: int
    """Compteur d'erreurs pour le mécanisme de fallback (max 3 tentatives)."""

    final_report: str
    """Rapport final Markdown généré par node_export_report."""

    messages: Annotated[list, operator.add]
    """Historique des messages du système (ajout cumulatif via operator.add)."""


print("AgentState TypedDict défini avec les champs :")
for field, annotation in AgentState.__annotations__.items():
    print(f"  {field}: {annotation}")

AgentState TypedDict défini avec les champs :
  domain: <class 'str'>
  ingestion_result: <class 'dict'>
  synthesis_result: <class 'dict'>
  gaps_result: <class 'dict'>
  human_approved: <class 'bool'>
  error_count: <class 'int'>
  final_report: <class 'str'>
  messages: typing.Annotated[list, <built-in function add>]


In [3]:
# ============================================================
# Fonctions des agents (wrappers pour les notebooks agents)
# ============================================================
# Chaque nœud LangGraph est une fonction Python qui reçoit l'état
# actuel et retourne un dict avec les champs mis à jour.
# ============================================================

# --- Imports des fonctions des agents ---
# Les fonctions des agents sont réimportées ici pour l'orchestration.
# Dans un environnement de production, elles seraient des modules Python séparés.

# Agent 1 – Ingestion
import pdfplumber
import chromadb
from chromadb.utils import embedding_functions
from llm_adapter import GeminiEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Constantes partagées
PAPERS_DIR      = "papers"
CHROMA_DB_PATH  = "../chroma_db"
COLLECTION_NAME = "literature_review"
CHUNK_SIZE      = 800
CHUNK_OVERLAP   = 150
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
BATCH_SIZE      = 50
SYNTHESIS_MODEL = "claude-sonnet-4-5"
GAP_MODEL       = "claude-sonnet-4-5"
N_RESULTS       = 10

# Initialisation partagée ChromaDB
embedding_fn = GeminiEmbeddingFunction()
chroma_client_orch = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection_orch = chroma_client_orch.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn
)

# Client Anthropic
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("Ressources partagées initialisées.")
print(f"ChromaDB : {collection_orch.count()} documents indexés")
print(f"Anthropic client : {SYNTHESIS_MODEL}")

Ressources partagées initialisées.
ChromaDB : 258 documents indexés
Anthropic client : claude-sonnet-4-5


In [4]:
# ============================================================
# Fonctions utilitaires des agents (réutilisées dans les nœuds)
# ============================================================

from pathlib import Path as _Path
import re as _re

# ---- Agent 1 helpers ----

class MCPFilesystemServer:
    """Filesystem MCP Server – abstraction layer conformant to MCP standard."""
    def __init__(self, base_path: str):
        self.base_path = _Path(base_path).resolve()
        self.base_path.mkdir(parents=True, exist_ok=True)
    def read_file(self, relative_path: str) -> bytes:
        return (self.base_path / relative_path).read_bytes()
    def list_directory(self, relative_path: str = ".") -> list:
        target = self.base_path / relative_path
        if not target.exists():
            return []
        return [f.name for f in target.iterdir()]
    def write_file(self, relative_path: str, content: str):
        p = self.base_path / relative_path
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
    def get_absolute_path(self, relative_path: str) -> _Path:
        return self.base_path / relative_path


def _extract_text_from_pdf(pdf_path: _Path) -> dict:
    """Extrait le texte d'un PDF via pdfplumber."""
    result = {
        "full_text": "",
        "metadata": {"filename": pdf_path.name, "total_pages": 0,
                     "extracted_at": datetime.now().isoformat()},
        "success": False, "error": None
    }
    try:
        pages_text = []
        with pdfplumber.open(pdf_path) as pdf:
            result["metadata"]["total_pages"] = len(pdf.pages)
            for i, page in enumerate(pdf.pages):
                txt = page.extract_text()
                if txt:
                    pages_text.append(f"[Page {i+1}]\n{txt.replace(chr(0), '').strip()}")
        result["full_text"] = "\n\n".join(pages_text)
        result["success"] = True
    except Exception as e:
        result["error"] = str(e)
    return result


def _chunk_text(text: str, metadata: dict) -> list:
    """Découpe le texte en chunks avec LangChain."""
    if not text or not text.strip():
        return []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""])
    raw_chunks = splitter.split_text(text)
    return [
        {"text": c.strip(), "metadata": {**metadata, "chunk_index": i,
                                          "total_chunks": len(raw_chunks)}}
        for i, c in enumerate(raw_chunks) if len(c.strip()) >= 50
    ]


def _ingest_paper(pdf_path: _Path) -> dict:
    """Ingère un PDF dans ChromaDB."""
    paper_id = pdf_path.stem.lower().replace(" ", "_").replace("-", "_")
    result = {"paper_id": paper_id, "filename": pdf_path.name,
              "chunks_created": 0, "total_pages": 0, "status": "pending", "error": None}
    try:
        extraction = _extract_text_from_pdf(pdf_path)
        if not extraction["success"] or not extraction["full_text"].strip():
            result["status"] = "error"
            result["error"] = extraction.get("error", "Empty text")
            return result
        result["total_pages"] = extraction["metadata"]["total_pages"]
        chunks = _chunk_text(extraction["full_text"], extraction["metadata"])
        if not chunks:
            result["status"] = "skipped"
            return result
        for batch_start in range(0, len(chunks), BATCH_SIZE):
            batch = chunks[batch_start: batch_start + BATCH_SIZE]
            collection_orch.upsert(
                ids=[f"{paper_id}_chunk_{c['metadata']['chunk_index']}" for c in batch],
                documents=[c["text"] for c in batch],
                metadatas=[{k: str(v) if not isinstance(v, (str, int, float, bool)) else v
                            for k, v in c["metadata"].items()} for c in batch]
            )
        result["chunks_created"] = len(chunks)
        result["status"] = "success"
    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)
    return result


# ---- Agent 2 helpers ----

SYNTHESIS_SYSTEM_PROMPT_ORCH = """You are an expert scientific literature analyst. Analyze the provided excerpts and synthesize them into a structured JSON with keys: themes, key_methodologies, main_findings, temporal_evolution, key_authors_concepts. Return ONLY valid JSON, no extra text."""


def _query_vector_db(query: str, n_results: int = 10) -> list:
    """Interroge ChromaDB pour les chunks pertinents."""
    total = collection_orch.count()
    if total == 0:
        return []
    n_results = min(n_results, total)
    try:
        raw = collection_orch.query(query_texts=[query], n_results=n_results,
                                    include=["documents", "metadatas", "distances"])
        return [{"text": d, "metadata": m, "distance": round(dist, 4)}
                for d, m, dist in zip(raw["documents"][0], raw["metadatas"][0], raw["distances"][0])]
    except Exception:
        return []


def _synthesize_theme(theme_query: str) -> dict:
    """Synthétise un thème via RAG + Claude."""
    chunks = _query_vector_db(theme_query, N_RESULTS)
    if not chunks:
        return {"theme": theme_query, "status": "no_data", "themes": [],
                "key_methodologies": [], "main_findings": [],
                "temporal_evolution": {"trend": "Données insuffisantes"}, "key_authors_concepts": []}
    context = "\n\n".join([f"--- Extrait #{i+1} (Source: {c['metadata'].get('filename','?')}) ---\n{c['text']}"
                            for i, c in enumerate(chunks)])
    user_msg = f"Research Theme: {theme_query}\n\nExcerpts:\n{context}\n\nSynthesize these excerpts."
    try:
        resp = anthropic_client.messages.create(
            model=SYNTHESIS_MODEL, max_tokens=2048,
            system=SYNTHESIS_SYSTEM_PROMPT_ORCH,
            messages=[{"role": "user", "content": user_msg}])
        raw = resp.content[0].text.strip()
        for prefix in ["```json", "```"]:
            if raw.startswith(prefix):
                raw = raw[len(prefix):]
        if raw.endswith("```"):
            raw = raw[:-3]
        synth = json.loads(raw.strip())
        synth.update({"theme": theme_query, "status": "success",
                      "input_tokens": resp.usage.input_tokens,
                      "output_tokens": resp.usage.output_tokens})
        return synth
    except Exception as e:
        return {"theme": theme_query, "status": "error", "error": str(e),
                "themes": [], "key_methodologies": [], "main_findings": [],
                "temporal_evolution": {"trend": "Erreur"}, "key_authors_concepts": []}


# ---- Agent 3 helpers ----

GAP_SYSTEM_PROMPT_ORCH = """You are an expert academic researcher identifying research gaps. Analyze the synthesis and find Methodological, Domain, Temporal, and Application gaps. Reason step-by-step inside <analysis> tags, then output structured JSON inside <gaps> tags with key 'research_gaps' as a list of gap objects (gap_id, gap_type, title, description, evidence, justification, suggested_research_direction, estimated_impact, feasibility, related_themes), plus 'priority_ranking', 'cross_cutting_observations', 'recommended_next_steps'."""


def _detect_gaps(synthesis_text: str) -> dict:
    """Détecte les research gaps via Chain-of-Thought."""
    user_msg = f"Analyze this synthesis and identify research gaps:\n\n{synthesis_text}\n\nBegin step-by-step analysis:"
    try:
        resp = anthropic_client.messages.create(
            model=GAP_MODEL, max_tokens=4096,
            system=GAP_SYSTEM_PROMPT_ORCH,
            messages=[{"role": "user", "content": user_msg}])
        raw = resp.content[0].text
        analysis_m = _re.search(r"<analysis>(.*?)</analysis>", raw, _re.DOTALL)
        reasoning = analysis_m.group(1).strip() if analysis_m else ""
        gaps_m = _re.search(r"<gaps>(.*?)</gaps>", raw, _re.DOTALL)
        if gaps_m:
            gaps_text = gaps_m.group(1).strip()
            for prefix in ["```json", "```"]:
                if gaps_text.startswith(prefix):
                    gaps_text = gaps_text[len(prefix):]
            if gaps_text.endswith("```"):
                gaps_text = gaps_text[:-3]
            gaps_data = json.loads(gaps_text.strip())
        else:
            json_m = _re.search(r"\{[\s\S]*\"research_gaps\"[\s\S]*\}", raw)
            gaps_data = json.loads(json_m.group(0)) if json_m else {"research_gaps": []}
        return {"reasoning": reasoning, "gaps": gaps_data, "status": "success",
                "metadata": {"input_tokens": resp.usage.input_tokens,
                             "output_tokens": resp.usage.output_tokens}}
    except Exception as e:
        return {"reasoning": "", "gaps": {"research_gaps": [], "priority_ranking": [],
                "cross_cutting_observations": "", "recommended_next_steps": ""},
                "status": "error", "metadata": {"error": str(e)}}


print("Toutes les fonctions utilitaires des agents définies.")

Toutes les fonctions utilitaires des agents définies.


In [5]:
# ============================================================
# Fonctions run_* des agents (utilisées par les nœuds LangGraph)
# ============================================================

def run_ingestion_agent() -> dict:
    """Agent 1 : Ingestion complète des PDFs dans ChromaDB."""
    start = time.time()
    papers_path = _Path(PAPERS_DIR).resolve()
    mcp = MCPFilesystemServer(str(papers_path))
    pdf_files = [f for f in mcp.list_directory(".") if f.lower().endswith(".pdf")]

    if not pdf_files:
        print(f"[Agent 1] Aucun PDF trouvé dans {papers_path}")
        return {"status": "no_papers_found", "total_papers": 0,
                "successful": 0, "failed": 0, "total_chunks": 0, "papers": []}

    print(f"[Agent 1] {len(pdf_files)} PDFs à ingérer...")
    results = []
    for pdf_name in pdf_files:
        pdf_path = mcp.get_absolute_path(pdf_name)
        r = _ingest_paper(pdf_path)
        results.append(r)
        print(f"  {r['filename']} → {r['status']} ({r['chunks_created']} chunks)")

    successful = [r for r in results if r["status"] == "success"]
    total_chunks = sum(r["chunks_created"] for r in results)
    summary = {"status": "completed", "total_papers": len(pdf_files),
               "successful": len(successful), "failed": len(results) - len(successful),
               "total_chunks": total_chunks, "papers": results,
               "duration_seconds": round(time.time() - start, 2),
               "completed_at": datetime.now().isoformat()}

    # Sauvegarder le log
    _Path(LOGS_DIR).mkdir(parents=True, exist_ok=True)
    with open(_Path(LOGS_DIR) / "ingestion_log.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"[Agent 1] Terminé : {len(successful)}/{len(pdf_files)} articles | {total_chunks} chunks")
    return summary


def run_synthesis_agent() -> dict:
    """Agent 2 : Synthèse thématique RAG + Claude."""
    start = time.time()
    RESEARCH_THEMES = [
        "automated literature review methods",
        "research gap detection",
        "NLP for scientific text",
        "knowledge extraction from papers",
        "multi-agent systems for research"
    ]
    print(f"[Agent 2] Synthèse de {len(RESEARCH_THEMES)} thèmes...")

    theme_syntheses = []
    total_in, total_out = 0, 0

    for i, theme in enumerate(RESEARCH_THEMES):
        print(f"  [{i+1}/{len(RESEARCH_THEMES)}] {theme}")
        synth = _synthesize_theme(theme)
        theme_syntheses.append(synth)
        total_in  += synth.get("input_tokens", 0)
        total_out += synth.get("output_tokens", 0)
        if i < len(RESEARCH_THEMES) - 1:
            time.sleep(1)

    global_synthesis = {
        "status": "completed", "model_used": SYNTHESIS_MODEL,
        "themes_analyzed": RESEARCH_THEMES, "theme_syntheses": theme_syntheses,
        "usage_stats": {
            "total_themes": len(RESEARCH_THEMES),
            "successful_syntheses": sum(1 for s in theme_syntheses if s.get("status") == "success"),
            "total_input_tokens": total_in, "total_output_tokens": total_out,
            "duration_seconds": round(time.time() - start, 2)
        },
        "completed_at": datetime.now().isoformat()
    }

    _Path(LOGS_DIR).mkdir(parents=True, exist_ok=True)
    with open(_Path(LOGS_DIR) / "synthesis_result.json", "w", encoding="utf-8") as f:
        json.dump(global_synthesis, f, indent=2, ensure_ascii=False)
    print(f"[Agent 2] Terminé : {global_synthesis['usage_stats']['successful_syntheses']}/{len(RESEARCH_THEMES)} synthèses")
    return global_synthesis


def run_gap_detection_agent() -> dict:
    """Agent 3 : Détection des research gaps avec Chain-of-Thought."""
    print("[Agent 3] Détection des research gaps...")

    # Charger la synthèse
    synthesis_path = _Path(LOGS_DIR) / "synthesis_result.json"
    if synthesis_path.exists():
        with open(synthesis_path, "r", encoding="utf-8") as f:
            synthesis_data = json.load(f)
    else:
        synthesis_data = {"theme_syntheses": [], "themes_analyzed": []}

    # Construire le texte de synthèse
    parts = [f"Themes: {', '.join(synthesis_data.get('themes_analyzed', []))}\n"]
    for s in synthesis_data.get("theme_syntheses", []):
        if s.get("status") != "success":
            continue
        parts.append(f"\n### {s.get('theme', '')}")
        for f_item in s.get("main_findings", []):
            parts.append(f"- {f_item.get('finding', '')}")
        for m in s.get("key_methodologies", []):
            parts.append(f"  Method: {m.get('method', '')}")
        t = s.get("temporal_evolution", {})
        if t.get("trend"):
            parts.append(f"  Trend: {t['trend']}")

    synthesis_text = "\n".join(parts)
    if len(synthesis_text.strip()) < 50:
        synthesis_text = "Domain: automated literature review and research gap detection. Limited data available."

    gaps_result = _detect_gaps(synthesis_text)

    full_result = {
        "status": gaps_result.get("status"),
        "gaps": gaps_result.get("gaps", {}),
        "reasoning_excerpt": gaps_result.get("reasoning", "")[:500],
        "metadata": gaps_result.get("metadata", {}),
        "generated_at": datetime.now().isoformat()
    }

    _Path(LOGS_DIR).mkdir(parents=True, exist_ok=True)
    with open(_Path(LOGS_DIR) / "gaps_result.json", "w", encoding="utf-8") as f:
        json.dump(full_result, f, indent=2, ensure_ascii=False)

    n_gaps = len(full_result["gaps"].get("research_gaps", []))
    print(f"[Agent 3] Terminé : {n_gaps} gaps identifiés")
    return full_result


print("Fonctions run_* des agents définies.")

Fonctions run_* des agents définies.


In [6]:
# ============================================================
# Nœuds LangGraph – Wrappers des agents
# ============================================================
# Chaque nœud reçoit AgentState et retourne un dict partiel
# avec uniquement les champs mis à jour.
# ============================================================

def node_ingestion(state: AgentState) -> dict:
    """Nœud LangGraph : Agent 1 – Ingestion des PDFs.
    
    Exécute l'ingestion complète et met à jour ingestion_result dans l'état.
    """
    print("\n" + "=" * 50)
    print("NŒUD : INGESTION")
    print("=" * 50)
    print(f"Domaine de recherche : {state.get('domain', 'N/A')}")

    try:
        ingestion_result = run_ingestion_agent()
        message = f"[{datetime.now().isoformat()}] Agent 1 complété : {ingestion_result.get('total_chunks', 0)} chunks indexés"
        return {
            "ingestion_result": ingestion_result,
            "messages": [message]
        }
    except Exception as e:
        error_msg = f"[{datetime.now().isoformat()}] Agent 1 ERREUR : {str(e)}"
        print(f"[ERREUR] node_ingestion : {e}")
        return {
            "ingestion_result": {"status": "error", "error": str(e), "total_chunks": 0},
            "error_count": state.get("error_count", 0) + 1,
            "messages": [error_msg]
        }


def node_synthesis(state: AgentState) -> dict:
    """Nœud LangGraph : Agent 2 – Synthèse thématique RAG.
    
    Exécute la synthèse et met à jour synthesis_result dans l'état.
    """
    print("\n" + "=" * 50)
    print("NŒUD : SYNTHÈSE")
    print("=" * 50)

    ingestion_result = state.get("ingestion_result", {})
    total_chunks = ingestion_result.get("total_chunks", 0)
    print(f"Chunks disponibles dans ChromaDB : {collection_orch.count()} (ingérés : {total_chunks})")

    try:
        synthesis_result = run_synthesis_agent()
        successful = synthesis_result.get("usage_stats", {}).get("successful_syntheses", 0)
        message = f"[{datetime.now().isoformat()}] Agent 2 complété : {successful} synthèses réussies"
        return {
            "synthesis_result": synthesis_result,
            "messages": [message]
        }
    except Exception as e:
        error_msg = f"[{datetime.now().isoformat()}] Agent 2 ERREUR : {str(e)}"
        print(f"[ERREUR] node_synthesis : {e}")
        return {
            "synthesis_result": {"status": "error", "error": str(e)},
            "error_count": state.get("error_count", 0) + 1,
            "messages": [error_msg]
        }


def node_gap_detection(state: AgentState) -> dict:
    """Nœud LangGraph : Agent 3 – Détection des research gaps.
    
    Exécute la détection CoT et met à jour gaps_result dans l'état.
    """
    print("\n" + "=" * 50)
    print("NŒUD : DÉTECTION DE GAPS")
    print("=" * 50)

    synthesis_result = state.get("synthesis_result", {})
    successful = synthesis_result.get("usage_stats", {}).get("successful_syntheses", 0)
    print(f"Synthèses disponibles : {successful}")

    try:
        gaps_result = run_gap_detection_agent()
        n_gaps = len(gaps_result.get("gaps", {}).get("research_gaps", []))
        message = f"[{datetime.now().isoformat()}] Agent 3 complété : {n_gaps} gaps identifiés"
        return {
            "gaps_result": gaps_result,
            "messages": [message]
        }
    except Exception as e:
        error_msg = f"[{datetime.now().isoformat()}] Agent 3 ERREUR : {str(e)}"
        print(f"[ERREUR] node_gap_detection : {e}")
        return {
            "gaps_result": {"status": "error", "error": str(e),
                            "gaps": {"research_gaps": []}},
            "error_count": state.get("error_count", 0) + 1,
            "messages": [error_msg]
        }


def node_human_validation(state: AgentState) -> dict:
    """Nœud LangGraph : Human-in-the-Loop – Validation des gaps.
    
    Affiche un résumé des gaps détectés et demande une validation
    interactive à l'utilisateur avant de procéder à l'export.
    """
    print("\n" + "=" * 50)
    print("NŒUD : VALIDATION HUMAINE (HITL)")
    print("=" * 50)

    gaps_result = state.get("gaps_result", {})
    research_gaps = gaps_result.get("gaps", {}).get("research_gaps", [])

    print(f"\n{len(research_gaps)} research gaps détectés :")
    print("-" * 40)
    for gap in research_gaps:
        gap_id   = gap.get("gap_id", "N/A")
        gap_type = gap.get("gap_type", "N/A")
        title    = gap.get("title", "N/A")
        impact   = gap.get("estimated_impact", "N/A")
        print(f"  {gap_id} [{gap_type}] – {title} (Impact: {impact})")

    observations = gaps_result.get("gaps", {}).get("cross_cutting_observations", "")
    if observations:
        print(f"\nObservations transversales : {observations[:200]}...")

    print("\n" + "-" * 40)

    # Demande de validation à l'utilisateur
    try:
        user_input = input("Approuvez-vous les gaps détectés pour générer le rapport final ? [o/n] : ").strip().lower()
        approved = user_input in ["o", "oui", "y", "yes", "1"]
    except (EOFError, KeyboardInterrupt):
        # En mode non-interactif (ex: exécution automatisée), approuver par défaut
        print("[Mode non-interactif] Approbation automatique.")
        approved = True

    if approved:
        print("Validation approuvée. Génération du rapport final...")
        message = f"[{datetime.now().isoformat()}] HITL : Gaps approuvés par l'utilisateur"
    else:
        print("Validation refusée. Relancement de la détection de gaps...")
        message = f"[{datetime.now().isoformat()}] HITL : Gaps refusés, nouvelle tentative"

    return {
        "human_approved": approved,
        "messages": [message]
    }


def node_export_report(state: AgentState) -> dict:
    """Nœud LangGraph : Génération et export du rapport final.
    
    Compile tous les résultats des agents en un rapport Markdown
    final et le sauvegarde dans le dossier Rapport.
    """
    print("\n" + "=" * 50)
    print("NŒUD : EXPORT DU RAPPORT FINAL")
    print("=" * 50)

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    domain = state.get("domain", "N/A")
    ingestion = state.get("ingestion_result", {})
    synthesis = state.get("synthesis_result", {})
    gaps      = state.get("gaps_result", {})

    research_gaps = gaps.get("gaps", {}).get("research_gaps", [])
    themes_analyzed = synthesis.get("themes_analyzed", [])

    # Construction du rapport final
    lines = [
        f"# Rapport Final – Revue de Littérature et Détection de Research Gaps",
        "",
        f"**Domaine de recherche** : {domain}  ",
        f"**Date de génération** : {now}  ",
        f"**Statut** : Approuvé par validation humaine  ",
        "",
        "---",
        "",
        "## 1. Résumé de l'Ingestion",
        "",
        f"- **Articles indexés** : {ingestion.get('successful', 0)} / {ingestion.get('total_papers', 0)}",
        f"- **Chunks vectorisés** : {ingestion.get('total_chunks', 0)}",
        f"- **Base de données** : ChromaDB (`{COLLECTION_NAME}`)",
        f"- **Modèle d'embedding** : `{EMBEDDING_MODEL}`",
        "",
        "---",
        "",
        "## 2. Thèmes Analysés",
        ""
    ]

    for i, theme in enumerate(themes_analyzed, 1):
        lines.append(f"{i}. {theme}")

    # Synthèses par thème
    lines += ["", "---", "", "## 3. Synthèses Thématiques", ""]
    for synth in synthesis.get("theme_syntheses", []):
        if synth.get("status") != "success":
            continue
        lines.append(f"### {synth.get('theme', 'N/A')}")
        lines.append("")
        for finding in synth.get("main_findings", [])[:3]:
            lines.append(f"- {finding.get('finding', 'N/A')}")
        temporal = synth.get("temporal_evolution", {})
        if temporal.get("trend"):
            trend = temporal["trend"]
            lines.append(f"\n**Tendance** : {trend[:200]}")
        lines.append("")

    # Research Gaps
    lines += ["---", "", "## 4. Research Gaps Identifiés", ""]
    lines.append(f"**Total** : {len(research_gaps)} gaps identifiés")
    lines.append("")

    priority = gaps.get("gaps", {}).get("priority_ranking", [])
    if priority:
        lines.append("**Ordre de priorité** : " + " → ".join(priority[:5]))
        lines.append("")

    for gap in research_gaps:
        lines += [
            f"### {gap.get('gap_id', 'N/A')} – {gap.get('title', 'N/A')}",
            "",
            f"**Type** : {gap.get('gap_type', 'N/A')} | **Impact** : {gap.get('estimated_impact', 'N/A')} | **Faisabilité** : {gap.get('feasibility', 'N/A')}",
            "",
            gap.get("description", ""),
            "",
            f"**Direction suggérée** : {gap.get('suggested_research_direction', 'N/A')}",
            ""
        ]

    # Recommandations
    next_steps = gaps.get("gaps", {}).get("recommended_next_steps", "")
    lines += [
        "---",
        "",
        "## 5. Recommandations Stratégiques",
        "",
        next_steps if next_steps else "Voir le rapport détaillé dans `../logs/gap_report.md`.",
        "",
        "---",
        "",
        "## 6. Métadonnées du SMA",
        "",
        f"| Paramètre | Valeur |",
        f"|-----------|--------|",
        f"| Modèle Synthèse | `{SYNTHESIS_MODEL}` |",
        f"| Modèle Gaps | `{GAP_MODEL}` |",
        f"| Embedding | `{EMBEDDING_MODEL}` |",
        f"| Tokens synthèse (in) | {synthesis.get('usage_stats', {}).get('total_input_tokens', 0):,} |",
        f"| Tokens synthèse (out) | {synthesis.get('usage_stats', {}).get('total_output_tokens', 0):,} |",
        f"| Tokens gaps (in) | {gaps.get('metadata', {}).get('input_tokens', 0):,} |",
        f"| Tokens gaps (out) | {gaps.get('metadata', {}).get('output_tokens', 0):,} |",
        "",
        "---",
        "",
        f"*Rapport généré automatiquement par le SMA – {now}*"
    ]

    final_report = "\n".join(lines)

    # Sauvegarder le rapport
    rapport_path = _Path(RAPPORT_DIR)
    rapport_path.mkdir(parents=True, exist_ok=True)
    report_file = rapport_path / "rapport_final.md"

    with open(report_file, "w", encoding="utf-8") as f:
        f.write(final_report)

    print(f"Rapport final sauvegardé : {report_file}")
    print(f"Taille du rapport : {len(final_report)} caractères")

    message = f"[{datetime.now().isoformat()}] Rapport final généré ({len(final_report)} chars)"
    return {
        "final_report": final_report,
        "messages": [message]
    }


def node_fallback(state: AgentState) -> dict:
    """Nœud LangGraph : Gestion des erreurs et récupération.
    
    Ce nœud est activé lorsqu'une erreur est détectée dans le pipeline.
    Il incrémente le compteur d'erreurs et retourne des résultats partiels
    pour permettre la continuation du workflow.
    """
    print("\n" + "=" * 50)
    print("NŒUD : FALLBACK (Gestion d'Erreur)")
    print("=" * 50)

    error_count = state.get("error_count", 0) + 1
    print(f"Tentative {error_count}/3 de récupération...")

    # Afficher le contexte d'erreur
    ingestion = state.get("ingestion_result", {})
    synthesis = state.get("synthesis_result", {})
    gaps      = state.get("gaps_result", {})

    print(f"  Ingestion status : {ingestion.get('status', 'N/A')}")
    print(f"  Synthesis status : {synthesis.get('status', 'N/A')}")
    print(f"  Gaps status      : {gaps.get('status', 'N/A')}")

    # Construire une synthèse de fallback si absente
    fallback_synthesis = synthesis if synthesis else {
        "status": "fallback",
        "themes_analyzed": ["general literature review"],
        "theme_syntheses": [],
        "usage_stats": {"successful_syntheses": 0, "total_input_tokens": 0, "total_output_tokens": 0}
    }

    message = f"[{datetime.now().isoformat()}] Fallback activé (tentative {error_count})"
    print(f"Fallback complété. Nouvelle tentative de synthèse...")

    return {
        "error_count": error_count,
        "synthesis_result": fallback_synthesis,
        "messages": [message]
    }


print("Tous les nœuds LangGraph définis.")

Tous les nœuds LangGraph définis.


In [7]:
# ============================================================
# Fonctions de routage conditionnel
# ============================================================

def route_after_validation(state: AgentState) -> Literal["export", "gap_detection", "__end__"]:
    """Routage conditionnel après la validation humaine.
    
    Détermine la prochaine étape selon la décision de l'utilisateur
    et le nombre d'erreurs accumulées.
    
    Args:
        state: État actuel du SMA.
        
    Returns:
        - 'export'         : Si l'utilisateur a approuvé
        - 'gap_detection'  : Si refus ET moins de 3 tentatives
        - END              : Si refus ET trop d'erreurs (sécurité)
    """
    human_approved = state.get("human_approved", False)
    error_count    = state.get("error_count", 0)

    if human_approved:
        print("[Router] Approuvé → Export du rapport")
        return "export"
    elif error_count < 3:
        print(f"[Router] Refusé (tentative {error_count + 1}/3) → Nouvelle détection de gaps")
        return "gap_detection"
    else:
        print(f"[Router] Refusé après {error_count} tentatives → Arrêt du workflow")
        return END


def should_fallback(state: AgentState) -> Literal["continue", "fallback"]:
    """Détecte si un mécanisme de fallback doit être activé.
    
    Vérifie si les résultats des agents précédents indiquent
    une erreur critique nécessitant une reprise.
    
    Args:
        state: État actuel du SMA.
        
    Returns:
        - 'continue'  : Si tout semble normal
        - 'fallback'  : Si une erreur critique est détectée
    """
    synthesis = state.get("synthesis_result", {})
    error_count = state.get("error_count", 0)

    # Condition de fallback : synthèse vide/erreur ET moins de 3 tentatives
    synthesis_failed = (not synthesis or
                        synthesis.get("status") == "error" or
                        synthesis.get("usage_stats", {}).get("successful_syntheses", 0) == 0)

    if synthesis_failed and error_count < 3:
        print(f"[Router] Synthèse défaillante → Fallback activé")
        return "fallback"

    print("[Router] Pipeline nominal → Continuation")
    return "continue"


print("Fonctions de routage définies :")
print("  route_after_validation : export | gap_detection | END")
print("  should_fallback        : continue | fallback")

Fonctions de routage définies :
  route_after_validation : export | gap_detection | END
  should_fallback        : continue | fallback


In [8]:
# ============================================================
# Construction du graphe LangGraph
# ============================================================

# Créer le graphe avec l'état partagé AgentState
workflow = StateGraph(AgentState)

# ---- Ajout des nœuds ----
workflow.add_node("ingestion",         node_ingestion)
workflow.add_node("synthesis",         node_synthesis)
workflow.add_node("gap_detection",     node_gap_detection)
workflow.add_node("human_validation",  node_human_validation)
workflow.add_node("export_report",     node_export_report)
workflow.add_node("fallback",          node_fallback)

# ---- Point d'entrée ----
workflow.set_entry_point("ingestion")

# ---- Arêtes fixes ----
workflow.add_edge("ingestion",     "synthesis")
workflow.add_edge("synthesis",     "gap_detection")
workflow.add_edge("gap_detection", "human_validation")
workflow.add_edge("export_report", END)
workflow.add_edge("fallback",      "synthesis")   # Le fallback relance la synthèse

# ---- Arêtes conditionnelles ----
# Après validation humaine : approuver → export | refus → re-détecter | trop d'erreurs → END
workflow.add_conditional_edges(
    "human_validation",
    route_after_validation,
    {
        "export":         "export_report",
        "gap_detection":  "gap_detection",
        END:              END
    }
)

# ---- Compilation du graphe ----
app = workflow.compile(
    # Limite de récursion pour éviter les boucles infinies
    # (important avec le cycle human_validation → gap_detection → human_validation)
)

print("Graphe LangGraph compilé avec succès.")
print()
print("Nœuds du graphe :")
print("  ingestion → synthesis → gap_detection → human_validation")
print("  human_validation → [export_report → END | gap_detection (refus) | END (max erreurs)]")
print("  fallback → synthesis (reprise en cas d'erreur)")

Graphe LangGraph compilé avec succès.



Nœuds du graphe :
  ingestion → synthesis → gap_detection → human_validation
  human_validation → [export_report → END | gap_detection (refus) | END (max erreurs)]
  fallback → synthesis (reprise en cas d'erreur)


In [9]:
# ============================================================
# Exécution du SMA via l'Orchestrateur LangGraph
# ============================================================

print("=" * 60)
print("DÉMARRAGE DU SYSTÈME MULTI-AGENTS (SMA)")
print("=" * 60)
print(f"Heure de démarrage : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# État initial du système
initial_state = {
    "domain": "automated literature review and research gap detection",
    "ingestion_result": {},
    "synthesis_result": {},
    "gaps_result": {},
    "human_approved": False,
    "error_count": 0,
    "final_report": "",
    "messages": [f"[{datetime.now().isoformat()}] SMA initialisé pour le domaine : automated literature review"]
}

print(f"Domaine de recherche : {initial_state['domain']}")
print()

# Lancer l'orchestrateur
try:
    result = app.invoke(
        initial_state,
        config={"recursion_limit": 10}  # Protection contre les boucles infinies
    )

    print()
    print("=" * 60)
    print("SMA TERMINÉ")
    print("=" * 60)
    print(f"Heure de fin : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()

    # Résumé de l'exécution
    print("RÉSUMÉ DE L'EXÉCUTION :")
    print(f"  Articles ingérés     : {result.get('ingestion_result', {}).get('successful', 0)}"
          f" / {result.get('ingestion_result', {}).get('total_papers', 0)}")
    print(f"  Chunks indexés       : {result.get('ingestion_result', {}).get('total_chunks', 0)}")
    print(f"  Synthèses réussies   : {result.get('synthesis_result', {}).get('usage_stats', {}).get('successful_syntheses', 0)}")
    print(f"  Gaps identifiés      : {len(result.get('gaps_result', {}).get('gaps', {}).get('research_gaps', []))}")
    print(f"  Approbation HITL     : {result.get('human_approved', False)}")
    print(f"  Rapport généré       : {'Oui' if result.get('final_report') else 'Non'}")
    print(f"  Erreurs rencontrées  : {result.get('error_count', 0)}")
    print()

    # Historique des messages
    print("HISTORIQUE DU WORKFLOW :")
    for msg in result.get("messages", []):
        print(f"  {msg}")

except Exception as e:
    print(f"[ERREUR CRITIQUE] Le SMA a rencontré une erreur fatale : {e}")
    print("Vérifiez les logs et relancez le workflow.")
    result = {}

DÉMARRAGE DU SYSTÈME MULTI-AGENTS (SMA)
Heure de démarrage : 2026-05-15 12:17:00

Domaine de recherche : automated literature review and research gap detection


NŒUD : INGESTION
Domaine de recherche : automated literature review and research gap detection
[Agent 1] 5 PDFs à ingérer...


  attention_is_all_you_need.pdf → success (58 chunks)


  bert_pre_training.pdf → error (0 chunks)


  chain_of_thought_wei2022.pdf → success (193 chunks)


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


... (6 outputs left out)

  [4/5] knowledge extraction from papers


  [5/5] multi-agent systems for research


[Agent 2] Terminé : 0/5 synthèses

NŒUD : DÉTECTION DE GAPS
Synthèses disponibles : 0
[Agent 3] Détection des research gaps...


[Agent 3] Terminé : 0 gaps identifiés

NŒUD : VALIDATION HUMAINE (HITL)

0 research gaps détectés :
----------------------------------------

----------------------------------------
[ERREUR CRITIQUE] Le SMA a rencontré une erreur fatale : raw_input was called, but this frontend does not support input requests.
Vérifiez les logs et relancez le workflow.


## Visualisation du Graphe

LangGraph permet de visualiser le graphe de workflow. La méthode `print_ascii()` affiche une représentation ASCII du graphe avec tous les nœuds et arêtes.

Cette visualisation est utile pour :
- Vérifier la structure du graphe
- Identifier les cycles potentiels
- Documenter le workflow pour d'autres développeurs

In [10]:
# ============================================================
# Visualisation du graphe LangGraph
# ============================================================

print("Représentation ASCII du graphe LangGraph :")
print("=" * 60)

try:
    app.get_graph().print_ascii()
except Exception as e:
    # Fallback : affichage manuel si print_ascii n'est pas disponible
    print(f"Note : print_ascii() non disponible ({e})")
    print("Structure du graphe :")
    print()
    print("  START")
    print("    |")
    print("    v")
    print("  [ingestion]")
    print("    |")
    print("    v")
    print("  [synthesis] <──────────────────────┐")
    print("    |                                │")
    print("    v                                │")
    print("  [gap_detection] <──────────┐    [fallback]")
    print("    |                        │")
    print("    v                        │")
    print("  [human_validation]         │")
    print("    |                        │")
    print("    ├─── Approuvé ──────> [export_report] ──> END")
    print("    ├─── Refusé (< 3) ──────────────────────────┘")
    print("    └─── Refusé (≥ 3) ──> END")

print()
print("Nodes du graphe :")
try:
    graph_data = app.get_graph()
    for node in graph_data.nodes:
        print(f"  • {node}")
except Exception:
    nodes = ["ingestion", "synthesis", "gap_detection", "human_validation", "export_report", "fallback"]
    for node in nodes:
        print(f"  • {node}")

Représentation ASCII du graphe LangGraph :
Note : print_ascii() non disponible (Install grandalf to draw graphs: `pip install grandalf`.)
Structure du graphe :

  START
    |
    v
  [ingestion]
    |
    v
  [synthesis] <──────────────────────┐
    |                                │
    v                                │
  [gap_detection] <──────────┐    [fallback]
    |                        │
    v                        │
  [human_validation]         │
    |                        │
    ├─── Approuvé ──────> [export_report] ──> END
    ├─── Refusé (< 3) ──────────────────────────┘
    └─── Refusé (≥ 3) ──> END

Nodes du graphe :
  • __start__
  • ingestion
  • synthesis
  • gap_detection
  • human_validation
  • export_report
  • fallback
  • __end__


## Analyse de la Context Window

### Stratégie de Gestion de la Mémoire dans le SMA

#### Le Problème de la Context Window

Les LLMs ont une limite de tokens pour leur context window. Avec de grandes collections d'articles scientifiques, il est impossible de passer tous les textes directement au modèle. Notre SMA résout ce problème avec une architecture en couches :

#### Architecture de Stockage à Deux Niveaux

```
┌─────────────────────────────────────────────────────────────┐
│  STOCKAGE EXTERNE (illimité)                                │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────┐  │
│  │   ChromaDB   │  │  JSON Logs   │  │  Rapport Final   │  │
│  │  (chunks +   │  │ (synthèses,  │  │   (Markdown)     │  │
│  │  embeddings) │  │    gaps)     │  │                  │  │
│  └──────────────┘  └──────────────┘  └──────────────────┘  │
└─────────────────────────────────────────────────────────────┘
                           ↕ RAG ↕
┌─────────────────────────────────────────────────────────────┐
│  CONTEXT WINDOW LLM (limité : ~200K tokens pour Sonnet)     │
│  ┌────────────────────────────────────────────────────┐     │
│  │  System Prompt + Top-K chunks pertinents (RAG)    │     │
│  │  + Question/Tâche courante                         │     │
│  │  → Résumés et synthèses (jamais les textes bruts)  │     │
│  └────────────────────────────────────────────────────┘     │
└─────────────────────────────────────────────────────────────┘
```

#### Principes Clés

1. **RAG au lieu de stuffing** : On ne passe jamais tous les textes d'un coup. La recherche vectorielle sélectionne uniquement les 10 chunks les plus pertinents pour chaque requête.

2. **Persistance externe** : ChromaDB et les fichiers JSON servent de mémoire à long terme. L'état LangGraph (`AgentState`) ne contient que des résumés, pas les textes bruts.

3. **Chunking stratégique** : Les textes sont découpés en segments de 800 caractères avec 150 de chevauchement pour conserver le contexte sémantique sans dépasser la window.

4. **Résumés progressifs** : Chaque agent produit des résumés qui alimentent le suivant (synthèse → gaps), réduisant exponentiellement le volume d'information transmis.

5. **Annotations cumulatives** : Le champ `messages` dans `AgentState` utilise `operator.add` pour accumuler l'historique, mais ne contient que des logs courts, pas les données complètes.

#### Impact sur les coûts

Cette architecture garantit que chaque appel API Claude ne traite que ~10 chunks × ~800 chars ≈ 8,000 caractères de contexte par thème, bien en dessous des limites de tokens, tout en restant pertinent grâce à la recherche sémantique.